# EDA en Bronze

Realizado especificamente para la evaluacion de los datos antes de pasar a Silver.

## Estructura actual

In [0]:
%sql 
DESCRIBE weather.bronze.ana_rio_uruguai

In [0]:
%sql
DESCRIBE weather.bronze.metar

In [0]:
%sql
DESCRIBE weather.bronze.nivel_ana

## Cobertura Temporal

### ANA Nivel

In [0]:
%sql
SELECT
    MIN(to_timestamp(Data_Hora_Medicao)) AS inicio,
    MAX(to_timestamp(Data_Hora_Medicao)) AS fin,
    COUNT(*) AS total_registros
FROM weather.bronze.nivel_ana

In [0]:
%sql
SELECT
    YEAR(to_timestamp(Data_Hora_Medicao)) AS anio,
    COUNT(*) registros
FROM weather.bronze.nivel_ana
GROUP BY anio
ORDER BY anio

Tomaremos los registros Desde 1950 por completitud anual.

### METAR Temperaturas en Aeropuertos

In [0]:
%sql
SELECT
    MIN(from_unixtime(obsTime)) AS inicio,
    MAX(from_unixtime(obsTime)) AS fin,
    COUNT(*) registros
FROM weather.bronze.metar

In [0]:
%sql
SELECT
    icaoId,
    MIN(from_unixtime(obsTime)) AS inicio,
    MAX(from_unixtime(obsTime)) AS fin,
    COUNT(*) registros
FROM weather.bronze.metar
GROUP BY icaoId
ORDER BY icaoId

### ANA Lluvias registradas

In [0]:
%sql
SELECT
    MIN(to_timestamp(Data_Hora_Medicao)) inicio,
    MAX(to_timestamp(Data_Hora_Medicao)) fin,
    COUNT(*) registros
FROM weather.bronze.ana_rio_uruguai

In [0]:
%sql
SELECT
    codigoestacao,
    MIN(to_timestamp(Data_Hora_Medicao)) inicio,
    MAX(to_timestamp(Data_Hora_Medicao)) fin,
    COUNT(*) registros
FROM weather.bronze.ana_rio_uruguai
GROUP BY codigoestacao
ORDER BY registros DESC

## Frecuencia real de medición

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def analizar_frecuencia(df, timestamp_col, nombre):
    print(f"\n===== {nombre} =====")

    # Convertir a timestamp
    df = df.withColumn("ts", F.to_timestamp(F.col(timestamp_col)))

    # 🔹 Cobertura temporal
    cobertura = df.select(
        F.min("ts").alias("inicio"),
        F.max("ts").alias("fin"),
        F.count("*").alias("total_registros")
    ).collect()[0]

    print(f"Rango: {cobertura['inicio']} → {cobertura['fin']}")
    print(f"Total registros: {cobertura['total_registros']}")

    # 🔹 Días observados
    df = df.withColumn("dia", F.to_date("ts"))

    dias = df.select(F.countDistinct("dia")).collect()[0][0]
    print(f"Días observados: {dias}")

    # 🔹 Registros por día
    freq_dia = df.groupBy("dia").agg(F.count("*").alias("registros"))

    avg_reg = freq_dia.select(F.avg("registros")).collect()[0][0]
    print(f"Promedio registros/día: {round(avg_reg,2)}")

    # 🔹 Distribución de frecuencias
    print("\nDistribución de frecuencia (top 5):")
    freq_dist = (
        freq_dia.groupBy("registros")
        .count()
        .orderBy(F.desc("count"))
        .limit(5)
    )
    freq_dist.show()

    # 🔹 Intervalos entre mediciones
    window = Window.orderBy("ts")

    df_diff = df.withColumn(
        "prev_ts", F.lag("ts").over(window)
    ).withColumn(
        "diff_min",
        (F.col("ts").cast("long") - F.col("prev_ts").cast("long")) / 60
    )

    print("\nIntervalos entre mediciones (top 5):")
    df_diff.groupBy("diff_min") \
        .count() \
        .orderBy(F.desc("count")) \
        .limit(5) \
        .show()

In [0]:
df_nivel = spark.table("weather.bronze.nivel_ana")

analizar_frecuencia(
    df_nivel,
    "Data_Hora_Medicao",
    "Nivel del Río"
)

In [0]:
df_metar = spark.table("weather.bronze.metar") \
    .withColumn("ts", F.from_unixtime("obsTime"))

analizar_frecuencia(
    df_metar,
    "ts",
    "Temperatura METAR"
)

In [0]:
df_lluvia = spark.table("weather.bronze.ana_rio_uruguai")

analizar_frecuencia(
    df_lluvia,
    "Data_Hora_Medicao",
    "Lluvia ANA"
)

## Datos Faltantes

In [0]:
from pyspark.sql import functions as F

def resumen_faltantes_diarios(df, col_fecha, nombre_tabla):
    
    # Parseo de timestamp (ajustar formato si hace falta)
    df = df.withColumn("fecha_ts", F.to_timestamp(col_fecha))
    
    # Día
    df = df.withColumn("fecha", F.to_date("fecha_ts"))
    
    # Rango temporal
    rango = df.agg(
        F.min("fecha").alias("min_fecha"),
        F.max("fecha").alias("max_fecha")
    ).collect()[0]
    
    min_fecha = rango["min_fecha"]
    max_fecha = rango["max_fecha"]
    
    # Calendario completo
    calendario = spark.sql(f"""
        SELECT explode(sequence(to_date('{min_fecha}'), to_date('{max_fecha}'), interval 1 day)) as fecha
    """)
    
    # Días con datos
    dias_con_datos = df.select("fecha").distinct()
    
    # Faltantes
    faltantes = calendario.join(dias_con_datos, on="fecha", how="left_anti")
    
    # Métricas
    total_dias = calendario.count()
    dias_presentes = dias_con_datos.count()
    dias_faltantes = faltantes.count()
    
    print(f"\n===== {nombre_tabla} =====")
    print(f"Rango: {min_fecha} → {max_fecha}")
    print(f"Total días esperados: {total_dias}")
    print(f"Días con datos: {dias_presentes}")
    print(f"Días faltantes: {dias_faltantes}")
    print(f"% faltantes: {round((dias_faltantes/total_dias)*100, 2)}%")
    
    print("\nEjemplo días faltantes:")
    faltantes.orderBy("fecha").show(10, False)


# --- Aplicación a tus tablas ---

df_nivel = spark.table("weather.bronze.nivel_ana")
df_temp = spark.table("weather.bronze.metar")
df_lluvia = spark.table("weather.bronze.ana_rio_uruguai")

resumen_faltantes_diarios(df_nivel, "Data_Hora_Medicao", "Nivel Río")
resumen_faltantes_diarios(df_temp, "reportTime", "Temperatura METAR")
resumen_faltantes_diarios(df_lluvia, "Data_Hora_Medicao", "Lluvia ANA")

In [0]:
from pyspark.sql import functions as F

# Lista de estaciones objetivo
estaciones = ["2751083", "2751037", "2752032", "2751066", "2753044"]

df = spark.table("weather.bronze.ana_rio_uruguai")

# --- 1. Estaciones presentes en la tabla ---
presentes = (
    df.select("codigoestacao")
      .distinct()
      .filter(F.col("codigoestacao").isin(estaciones))
)

presentes_list = [row["codigoestacao"] for row in presentes.collect()]

# --- 2. Estaciones faltantes ---
faltantes = list(set(estaciones) - set(presentes_list))

print("Estaciones encontradas:", presentes_list)
print("Estaciones faltantes:", faltantes)

# --- 3. Conteo de registros por estación (sanity check) ---
print("\nCantidad de registros por estación:")
df.filter(F.col("codigoestacao").isin(estaciones)) \
  .groupBy("codigoestacao") \
  .count() \
  .orderBy(F.desc("count")) \
  .show()

## Duplicados

In [0]:
from pyspark.sql import functions as F

def analizar_duplicados(tabla, col_fecha, claves_extra=None):
    
    df = spark.table(tabla)
    
    # Parseo fecha
    df = df.withColumn("fecha_ts", F.to_timestamp(col_fecha))
    
    # --- 1. Duplicados exactos ---
    total = df.count()
    distinct = df.dropDuplicates().count()
    dup_exactos = total - distinct
    
    # --- 2. Duplicados por clave lógica ---
    claves = ["fecha_ts"]
    if claves_extra:
        claves += claves_extra
    
    dup_keys = (
        df.groupBy(claves)
          .count()
          .filter(F.col("count") > 1)
    )
    
    cantidad_dup_keys = dup_keys.count()
    
    print(f"\n===== {tabla} =====")
    print(f"Total registros: {total}")
    print(f"Duplicados exactos: {dup_exactos} ({round((dup_exactos/total)*100,4)}%)")
    print(f"Claves duplicadas: {cantidad_dup_keys}")
    
    print("\nEjemplo duplicados por clave:")
    dup_keys.orderBy(F.desc("count")).show(5, False)


# --- Ejecutar para tus tablas ---

analizar_duplicados(
    "weather.bronze.ana_rio_uruguai",
    "Data_Hora_Medicao",
    ["codigoestacao"]
)

analizar_duplicados(
    "weather.bronze.nivel_ana",
    "Data_Hora_Medicao",
    ["codigoestacao"]
)

analizar_duplicados(
    "weather.bronze.metar",
    "reportTime",
    ["icaoId"]
)

## Outliers

In [0]:
from pyspark.sql import functions as F

def analizar_outliers(tabla, col_valor, col_fecha, col_grupo, nombre):
    
    df = spark.table(tabla) \
        .withColumn("valor", F.col(col_valor).cast("double")) \
        .withColumn("fecha_ts", F.to_timestamp(col_fecha)) \
        .filter(F.col("valor").isNotNull())
    
    # Cuartiles globales (rápido, suficiente para EDA)
    q1, q3 = df.approxQuantile("valor", [0.25, 0.75], 0.01)
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    df_out = df.withColumn(
        "es_outlier",
        (F.col("valor") < lower) | (F.col("valor") > upper)
    )
    
    total = df_out.count()
    outliers = df_out.filter("es_outlier").count()
    
    print(f"\n===== {nombre} =====")
    print(f"Q1: {q1} | Q3: {q3} | IQR: {iqr}")
    print(f"Lower: {lower} | Upper: {upper}")
    print(f"Total: {total}")
    print(f"Outliers: {outliers} ({round((outliers/total)*100,2)}%)")
    
    print("\nOutliers por grupo:")
    df_out.groupBy(col_grupo) \
        .agg(F.sum(F.col("es_outlier").cast("int")).alias("outliers")) \
        .orderBy(F.desc("outliers")) \
        .show(10)
    
    print("\nEjemplo outliers:")
    df_out.filter("es_outlier") \
        .select(col_grupo, "fecha_ts", "valor") \
        .orderBy(F.desc("valor")) \
        .show(10, False)

In [0]:
analizar_outliers(
    "weather.bronze.nivel_ana",
    "Cota_Adotada",
    "Data_Hora_Medicao",
    "codigoestacao",
    "Nivel del Río"
)

In [0]:
analizar_outliers(
    "weather.bronze.ana_rio_uruguai",
    "REPLACE(Chuva_Adotada, ',', '.')",
    "Data_Hora_Medicao",
    "codigoestacao",
    "Lluvia"
)

In [0]:
analizar_outliers(
    "weather.bronze.metar",
    "tmpf",   # o temp si preferís
    "reportTime",
    "icaoId",
    "Temperatura"
)

## Analisis de Frecuencia

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def analizar_frecuencia(tabla, col_fecha, col_grupo, nombre):
    
    df = spark.table(tabla) \
        .withColumn("fecha_ts", F.to_timestamp(col_fecha)) \
        .filter(F.col("fecha_ts").isNotNull())
    
    w = Window.partitionBy(col_grupo).orderBy("fecha_ts")
    
    df_diff = df.withColumn(
        "diff_min",
        (F.col("fecha_ts").cast("long") - F.lag("fecha_ts").over(w).cast("long")) / 60
    )
    
    df_diff = df_diff.filter(F.col("diff_min").isNotNull())
    
    print(f"\n===== {nombre} =====")
    
    # --- Resumen global ---
    df_diff.agg(
        F.min("diff_min").alias("min_minutos"),
        F.expr("percentile_approx(diff_min, 0.5)").alias("mediana_minutos"),
        F.max("diff_min").alias("max_minutos")
    ).show()
    
    # --- Frecuencia dominante (top valores) ---
    print("\nFrecuencias más comunes (minutos):")
    df_diff.groupBy("diff_min") \
        .count() \
        .orderBy(F.desc("count")) \
        .show(10)
    
    # --- Por grupo ---
    print("\nFrecuencia por grupo:")
    df_diff.groupBy(col_grupo) \
        .agg(
            F.expr("percentile_approx(diff_min, 0.5)").alias("mediana_min"),
            F.min("diff_min").alias("min_min"),
            F.max("diff_min").alias("max_min"),
            F.count("*").alias("n_registros")
        ) \
        .orderBy(F.desc("n_registros")) \
        .show(10)

In [0]:
analizar_frecuencia(
    "weather.bronze.nivel_ana",
    "Data_Hora_Medicao",
    "codigoestacao",
    "Nivel del Río"
)

In [0]:
analizar_frecuencia(
    "weather.bronze.ana_rio_uruguai",
    "Data_Hora_Medicao",
    "codigoestacao",
    "Lluvia"
)

In [0]:
analizar_frecuencia(
    "weather.bronze.metar",
    "reportTime",
    "icaoId",
    "Temperatura"
)